In [1]:
import pandas as pd 
import numpy as np

In [2]:
#Urban Structure → Heat Amplification → Energy Stress → Health Impact

In [3]:
df_temp = pd.read_csv("temperature_clean.csv", parse_dates=["date"])
df_surface = pd.read_csv("urban_surface_clean.csv")

df_struct_temp = df_temp.merge(df_surface, on="neighbourhood_id", how="left")

In [9]:
df_struct_temp.columns

Index(['neighbourhood_id', 'date', 'avg_temp', 'max_temp', 'night_temp',
       'surface_temp', 'humidity', 'wind_speed', 'solar_radiation',
       'urban_heat_index', 'latitude', 'longitude', 'distance_from_center_km',
       'tree_cover_pct', 'asphalt_pct', 'building_density', 'median_income',
       'population_density', 'heat_retention_factor',
       'infrastructure_quality_index', 'social_vulnerability_index'],
      dtype='object')

In [6]:
(df_struct_temp["tree_cover_pct_x"] - 
 df_struct_temp["tree_cover_pct_y"]).abs().sum()

np.float64(0.0)

In [7]:
cols_to_drop = [col for col in df_struct_temp.columns if col.endswith("_x")]
df_struct_temp = df_struct_temp.drop(columns=cols_to_drop)

In [8]:
df_struct_temp.columns = df_struct_temp.columns.str.replace("_y", "", regex=False)

In [14]:
import statsmodels.api as sm

X = df_struct_temp[[
    "tree_cover_pct",
    "asphalt_pct",
    "building_density",
    "heat_retention_factor"
]]

X = sm.add_constant(X)
y = df_struct_temp["surface_temp"]

model_heat = sm.OLS(y, X).fit()
print(model_heat.summary())

                            OLS Regression Results                            
Dep. Variable:           surface_temp   R-squared:                       0.132
Model:                            OLS   Adj. R-squared:                  0.132
Method:                 Least Squares   F-statistic:                 3.793e+04
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:07:50   Log-Likelihood:            -3.3879e+06
No. Observations:             1000000   AIC:                         6.776e+06
Df Residuals:                  999995   BIC:                         6.776e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    25.68

In [ ]:
#Improve Heat Model (Add Weather Controls)

In [15]:
import statsmodels.api as sm

X = df_struct_temp[[
    "tree_cover_pct",
    "asphalt_pct",
    "building_density",
    "heat_retention_factor",
    "solar_radiation",
    "wind_speed",
    "humidity",
    "avg_temp"
]]

X = sm.add_constant(X)
y = df_struct_temp["surface_temp"]

model_heat_control = sm.OLS(y, X).fit()
print(model_heat_control.summary())

                            OLS Regression Results                            
Dep. Variable:           surface_temp   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 3.177e+30
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:07:57   Log-Likelihood:             2.5790e+07
No. Observations:             1000000   AIC:                        -5.158e+07
Df Residuals:                  999991   BIC:                        -5.158e+07
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                  2.028e-

In [ ]:
#Quantify Pure Structural Effect

In [12]:
print("R-squared:", model_heat_control.rsquared)

R-squared: 1.0


In [ ]:
#Move to Energy Demand (Lag-Based Model)

In [13]:
df_elec = pd.read_csv("electricity_clean.csv", parse_dates=["date"])

df_elec = df_elec.dropna(subset=["avg_temp_lag2"])

X = df_elec[[
    "avg_temp_lag2",
    "population_density",
    "median_income"
]]

X = sm.add_constant(X)
y = df_elec["residential_kwh"]

model_energy = sm.OLS(y, X).fit()
print(model_energy.summary())

                            OLS Regression Results                            
Dep. Variable:        residential_kwh   R-squared:                       0.776
Model:                            OLS   Adj. R-squared:                  0.776
Method:                 Least Squares   F-statistic:                 1.149e+06
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:07:12   Log-Likelihood:            -6.9198e+06
No. Observations:              997000   AIC:                         1.384e+07
Df Residuals:                  996996   BIC:                         1.384e+07
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const               1198.0232      1

In [17]:
#Health Model (Lagged)

In [18]:
#3-day lag for fatigue:
df_health = pd.read_csv("health_clean.csv", parse_dates=["date"])
df_health3 = df_health.dropna(subset=["avg_temp_lag3"])

X = df_health3[[
    "avg_temp_lag3",
    "social_vulnerability_index"
]]

X = sm.add_constant(X)
y = df_health3["heat_fatigue_cases"]

model_health3 = sm.OLS(y, X).fit()
print(model_health3.summary())

                            OLS Regression Results                            
Dep. Variable:     heat_fatigue_cases   R-squared:                       0.732
Model:                            OLS   Adj. R-squared:                  0.732
Method:                 Least Squares   F-statistic:                 1.367e+06
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:10:05   Log-Likelihood:            -3.2639e+06
No. Observations:              998500   AIC:                         6.528e+06
Df Residuals:                  998497   BIC:                         6.528e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               

In [19]:
#5-day lag for deaths:

In [20]:
df_health5 = df_health.dropna(subset=["avg_temp_lag5"])

X = df_health5[[
    "avg_temp_lag5",
    "social_vulnerability_index"
]]

X = sm.add_constant(X)
y = df_health5["heatstroke_deaths"]

model_health5 = sm.OLS(y, X).fit()
print(model_health5.summary())

                            OLS Regression Results                            
Dep. Variable:      heatstroke_deaths   R-squared:                       0.698
Model:                            OLS   Adj. R-squared:                  0.698
Method:                 Least Squares   F-statistic:                 1.152e+06
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:14:40   Log-Likelihood:            -1.1630e+06
No. Observations:              997500   AIC:                         2.326e+06
Df Residuals:                  997497   BIC:                         2.326e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               

In [21]:
#The Advanced Move

In [22]:
df_full = df_health.merge(df_struct_temp, on=["neighbourhood_id","date"], how="left")

In [23]:
df_full = df_full.dropna(subset=["avg_temp_lag3"])

X = df_full[[
    "tree_cover_pct",
    "asphalt_pct",
    "heat_retention_factor",
    "avg_temp_lag3"
]]

X = sm.add_constant(X)
y = df_full["heat_fatigue_cases"]

model_full = sm.OLS(y, X).fit()
print(model_full.summary())

                            OLS Regression Results                            
Dep. Variable:     heat_fatigue_cases   R-squared:                       0.622
Model:                            OLS   Adj. R-squared:                  0.622
Method:                 Least Squares   F-statistic:                 4.107e+05
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:15:49   Log-Likelihood:            -3.4365e+06
No. Observations:              998500   AIC:                         6.873e+06
Df Residuals:                  998495   BIC:                         6.873e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                   -42.99